# ML-KEM-768 Benchmark Notebook

This notebook measures:
- Hardware cycles per operation (median/min/max)
- Hardware latency in microseconds from cycle counter
- Wall time seen by Python (includes register writes, cache operations, polling)
- Throughput in ops/s under pulse-then-poll control

In [ ]:
import os
import secrets
import statistics
import time

from ml_kem_driver import MLKem768, cycles_to_us


def bench_op(label, fn, n):
    cycles_list = []
    wall_list = []
    for _ in range(n):
        t0 = time.monotonic()
        result = fn()
        wall_list.append(time.monotonic() - t0)
        cyc = result[-1] if isinstance(result, tuple) else result
        cycles_list.append(cyc)

    c_med = statistics.median(cycles_list)
    c_min = min(cycles_list)
    c_max = max(cycles_list)
    w_med = statistics.median(wall_list)
    w_min = min(wall_list)
    w_max = max(wall_list)
    throughput = n / sum(wall_list) if sum(wall_list) > 0 else 0.0

    print(f"{label}")
    print(f"  HW cycles  : median={int(c_med)} min={c_min} max={c_max}")
    print(f"  HW latency : median={cycles_to_us(c_med):.1f} us")
    print(f"  Wall time  : median={w_med*1e6:.1f} us min={w_min*1e6:.1f} us max={w_max*1e6:.1f} us")
    print(f"  SW overhead: ~{(w_med*1e6 - cycles_to_us(c_med)):.1f} us")
    print(f"  Throughput : {throughput:.1f} ops/s")

    return {
        "label": label,
        "cycles": cycles_list,
        "wall_s": wall_list,
        "cycle_median": c_med,
        "wall_median_s": w_med,
        "throughput_ops_s": throughput,
    }


In [ ]:
bitfile = os.environ.get("ML_KEM_BIT", "./ml_kem.bit")
n = 10  # increase to 100 or 500

print(f"bitfile: {bitfile}")
print(f"iterations per op: {n}")


In [ ]:
kem = MLKem768(bitfile)

# Warm-up keypair for encaps/decaps benches
warm_d = secrets.token_bytes(32)
warm_z = secrets.token_bytes(32)
pk_warm, sk_warm, warm_cycles = kem.keygen(warm_d, warm_z)
print(f"Warm-up keygen cycles: {warm_cycles}")


In [ ]:
res_keygen = bench_op(
    "KeyGen (random d,z)",
    lambda: kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32)),
    n,
)


In [ ]:
res_encaps = bench_op(
    "Encaps (fixed pk, random m)",
    lambda: kem.encaps(pk_warm, secrets.token_bytes(32)),
    n,
)


In [ ]:
ct_warm, ss_warm, cyc_warm_enc = kem.encaps(pk_warm, secrets.token_bytes(32))
res_decaps = bench_op(
    "Decaps (fixed sk, fixed valid ct)",
    lambda: kem.decaps(sk_warm, ct_warm),
    n,
)


In [ ]:
def full_kem_once():
    d = secrets.token_bytes(32)
    z = secrets.token_bytes(32)
    m = secrets.token_bytes(32)
    pk, sk, c1 = kem.keygen(d, z)
    ct, ss1, c2 = kem.encaps(pk, m)
    ss2, c3 = kem.decaps(sk, ct)
    if ss1 != ss2:
        raise RuntimeError("Round-trip ss mismatch")
    return (c1 + c2 + c3,)

res_full = bench_op("Full KEM (KG + Enc + Dec)", full_kem_once, n)

kem.close()


## How to interpret results

- `HW latency` comes from the on-chip cycle counter and reflects accelerator compute time.
- `Wall time` includes Python/PYNQ overhead (register writes, flush/invalidate, polling loop).
- If wall-hw gap grows too much, optimize software control path first; RTL may already be fine.
- Compare `res_full` against your expected end-to-end budget for deployment.